# Multi-Source Merging & Join Validation

This notebook demonstrates systematic multi-source data merging and join validation techniques.

### Tasks Covered:
1. **Explicit Join with Row Count Validation**: Comparing pre and post-merge row counts.
2. **Detect Unmatched Keys**: Isolating unmatched left keys and orphaned right keys.
3. **Compare Join Types**: Benchmarking Inner, Left, Right, and Outer joins.
4. **Validate No Unexpected Duplication**: Checking column suffix conflicts and key multiplicity.
5. **Document Join Decision**: Saving business audit report to `output/join_decision_report.json`.

In [ ]:
import pandas as pd
import numpy as np
import json
import os

# Load raw datasets
df_customers = pd.read_csv('../data/raw/join_customers.csv')
df_orders = pd.read_csv('../data/raw/join_orders.csv')

print(f"Customers shape: {df_customers.shape}")
print(f"Orders shape: {df_orders.shape}")

## Task 1: Explicit Join with Row Count Validation

Perform an explicit Left join on `customer_id` and track row count differences.

In [ ]:
print(f"Left: {len(df_customers)}")
print(f"Right: {len(df_orders)}")

df_merged = pd.merge(df_customers, df_orders, on='customer_id', how='left')

print(f"Merged: {len(df_merged)}")
print(f"Change: {len(df_merged) - len(df_customers)}")

## Task 2: Detect Unmatched Keys

Isolate customers with no orders and orphaned orders referencing non-existent customers.

In [ ]:
unmatched_customers = df_customers[~df_customers['customer_id'].isin(df_orders['customer_id'])]
unmatched_orders = df_orders[~df_orders['customer_id'].isin(df_customers['customer_id'])]

print(f"Customers without orders: {len(unmatched_customers)}")
print(f"Orphaned orders: {len(unmatched_orders)}")

os.makedirs('../output', exist_ok=True)
unmatched_customers.to_csv('../output/unmatched_customers.csv', index=False)
unmatched_orders.to_csv('../output/unmatched_orders.csv', index=False)

## Task 3: Compare Join Types

Compare row counts across `inner`, `left`, and `outer` joins to understand data semantics.

In [ ]:
inner = pd.merge(df_customers, df_orders, on='customer_id', how='inner')
left = pd.merge(df_customers, df_orders, on='customer_id', how='left')
outer = pd.merge(df_customers, df_orders, on='customer_id', how='outer')

print(f"Inner: {len(inner)}, Left: {len(left)}, Outer: {len(outer)}")

## Task 4: Validate No Unexpected Duplication

Verify merge key cardinality and ensure no unexpected column collisions.

In [ ]:
# Check for unexpected column conflicts
print(df_merged.columns)

# Check max multiplicity per customer_id
key_counts = df_merged['customer_id'].value_counts()
print(f"Max orders per customer: {key_counts.max()}")

## Task 5: Document Join Decision

Create a structured JSON decision report for auditability.

In [ ]:
join_report = {
    'join_type': 'left',
    'left_table': 'customers',
    'right_table': 'orders',
    'join_key': 'customer_id',
    'left_rows': len(df_customers),
    'right_rows': len(df_orders),
    'result_rows': len(df_merged),
    'unmatched_left': len(unmatched_customers),
    'unmatched_right': len(unmatched_orders),
    'reasoning': 'Left join preserves all customers; unmatched customers have no orders'
}

print(json.dumps(join_report, indent=2))
with open('../output/join_decision_report.json', 'w') as f:
    json.dump(join_report, f, indent=2)